# S08. Tipos de Estudo
## Types of Studies

[◀ Anterior](S07_Parametros_Estatisticas.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S09_Tipos_de_Amostras.ipynb)

## 🎯 Objetivos
- Diferenciar estudos observacionais de experimentais
- Compreender os princípios de delineamento experimental
- Entender o papel da randomização e dos grupos de controle
- Saber identificar e evitar fatores de confusão em estudos

## 📝 Introdução

A forma como coletamos dados determina o que podemos concluir a partir deles. Estatísticos classificam os estudos em duas grandes categorias: **observacionais** (observational studies), onde apenas observamos e medimos sem intervir, e **experimentais** (experimental studies), onde ativamente manipulamos uma variável para observar seu efeito. Cada tipo tem seus pontos fortes, limitações e tipos de inferência que permite.

## 📚 Explicação Detalhada

### 1. Estudos Observacionais

Em estudos **observacionais**, o pesquisador não interfere — apenas coleta dados sobre os fenômenos como eles ocorrem naturalmente.

**Tipos:**
- **Transversal (cross-sectional)**: dados coletados em um único momento. Ex: pesquisa de satisfação aplicada em janeiro de 2024
- **Coorte (cohort)**: acompanha um grupo ao longo do tempo. Ex: acompanhar fumantes e não-fumantes por 20 anos para ver incidência de câncer
- **Caso-controle (case-control)**: seleciona indivíduos com (casos) e sem (controles) uma condição e olha para trás para identificar fatores. Ex: entrevistar pessoas com e sem câncer de pulmão sobre histórico de tabagismo

**Limitação principal:** não permitem estabelecer causalidade com certeza, pois podem existir **fatores de confusão** (confounding variables) não controlados.

### 2. Estudos Experimentais

Em estudos **experimentais**, o pesquisador manipula intencionalmente uma ou mais **variáveis independentes** (tratamentos) e mede o efeito nas **variáveis dependentes** (desfechos).

**Características essenciais:**
- **Manipulação**: o pesquisador decide quem recebe o quê
- **Randomização**: os sujeitos são alocados aleatoriamente aos grupos
- **Controle**: grupos de controle permitem comparar o efeito do tratamento
- **Réplica**: repetição do experimento para verificar consistência

**Exemplo clássico:** ensaio clínico randomizado para testar uma vacina. Metade dos voluntários recebe a vacina (grupo experimental), metade recebe placebo (grupo controle). A alocação é aleatória e duplo-cega.

### 3. Delineamento Experimental e Controle

O **delineamento experimental** (experimental design) é o plano do experimento. Um bom delineamento minimiza vieses e maximiza a capacidade de detectar efeitos reais.

**Princípios de Fisher (1925):**
1. **Randomização**: alocar tratamentos aleatoriamente elimina vieses de seleção
2. **Repetição**: múltiplas observações reduzem o erro aleatório
3. **Bloqueio**: agrupar unidades similares para controlar variabilidade

**Tipos de controle:**
- **Grupo controle**: não recebe o tratamento (ou recebe placebo)
- **Controle histórico**: compara com dados anteriores ao experimento
- **Controle ativo**: compara com um tratamento já estabelecido
- **Placebo**: substância inerte que parece com o tratamento real

In [ ]:
# Simulação de um experimento controlado randomizado
import numpy as np
from scipy import stats

np.random.seed(42)

# PARÂMETROS DO EXPERIMENTO
n_por_grupo = 50          # 50 voluntários em cada grupo

# Simulando 100 voluntários
# Variável de confusão: idade (afeta tanto grupos quanto desfecho)
idades = np.random.randint(20, 70, n_por_grupo * 2)

# ALOCAÇÃO ALEATÓRIA (randomização)
# Embaralhamos os índices para simular randomização
indices = np.arange(n_por_grupo * 2)
np.random.shuffle(indices)
grupo = np.array(['Controle'] * n_por_grupo + ['Tratamento'] * n_por_grupo)
grupo = grupo[indices]
idades = idades[indices]

# Função biológica simulada
# Grupo tratamento recebe um fármaco que reduz pressão arterial em média 10 mmHg
efeito_verdadeiro = -10  # mmHg

def pressao_arterial(idade, tratamento):
    # Pressão basal aumenta com idade
    basal = 120 + (idade - 20) * 0.3
    # Efeito do tratamento
    efeito = efeito_verdadeiro if tratamento else 0
    # Erro aleatório individual
    erro = np.random.normal(0, 8)
    return basal + efeito + erro

# Mensurando a pressão arterial
pressoes = np.array([
    pressao_arterial(idades[i], grupo[i] == 'Tratamento')
    for i in range(len(grupo))
])

# RESULTADOS
controle = pressoes[grupo == 'Controle']
tratamento = pressoes[grupo == 'Tratamento']

print('RESULTADOS DO EXPERIMENTO')
print(f'Grupo Controle:   média = {np.mean(controle):.1f} mmHg, n = {len(controle)}')
print(f'Grupo Tratamento: média = {np.mean(tratamento):.1f} mmHg, n = {len(tratamento)}')
print(f'Diferença observada: {np.mean(tratamento) - np.mean(controle):.1f} mmHg')
print(f'Diferença real esperada: {efeito_verdadeiro} mmHg')
print()

# Teste t para comparar grupos
t_stat, p_valor = stats.ttest_ind(tratamento, controle)
print(f'Teste t: t = {t_stat:.3f}, p-valor = {p_valor:.4f}')
print(f'Conclusão: ', 'Efeito estatisticamente significativo' if p_valor < 0.05 else 'Sem evidência de efeito')

### 4. Fatores de Confusão e Causalidade

Um **fator de confusão** (confounding variable) é uma variável que influencia tanto a exposição quanto o desfecho, criando uma associação espúria.

**Exemplo clássico:** consumo de sorvete e afogamentos são correlacionados. A variável de confusão é a temperatura — nos dias quentes, mais pessoas comem sorvete e mais pessoas vão à praia, aumentando afogamentos.

**Como controlar fatores de confusão:**
- **Randomização**: distribui fatores de confusão igualmente entre grupos
- **Estratificação**: analisar dentro de subgrupos homogêneos
- **Ajuste estatístico**: regressão multivariada para isolar efeitos

**Critérios de causalidade (Bradford Hill):**
1. Força da associação | 2. Consistência | 3. Especificidade
4. Temporalidade | 5. Gradiente biológico | 6. Plausibilidade
7. Coerência | 8. Evidência experimental | 9. Analogia

In [ ]:
# Demonstração de fator de confusão
import pandas as pd

np.random.seed(42)

n = 1000

# Fator de confusão: idade (quanto mais velho, maior
# probabilidade de tomar o medicamento e maior chance de complicação)
idade = np.random.uniform(30, 80, n)

# Probabilidade de tomar o medicamento aumenta com a idade
p_medicamento = 1 / (1 + np.exp(-(idade - 55) / 10))
toma_medicamento = np.random.binomial(1, p_medicamento)

# Probabilidade de complicação aumenta com idade
efeito_medicamento = -0.15  # medicamento reduz complicações
p_complicacao = 1 / (1 + np.exp(-(idade - 60) / 8 + toma_medicamento * efeito_medicamento))
complicacao = np.random.binomial(1, p_complicacao)

# Criando dataframe
dados = pd.DataFrame({'idade': idade, 'medicamento': toma_medicamento, 'complicacao': complicacao})

# Análise ingênua (sem considerar idade)
tabela = pd.crosstab(dados['medicamento'], dados['complicacao'], normalize='index')
print('Análise ingênua — taxa de complicação por grupo:')
print(tabela)
p_controle = dados[dados['medicamento'] == 0]['complicacao'].mean()
p_tratamento = dados[dados['medicamento'] == 1]['complicacao'].mean()
print(f'\nDiferença bruta: {p_tratamento - p_controle:.3f}')
print('(Parece que o medicamento AUMENTA o risco! — isso é o viés de confusão)')
print()

# Análise estratificada por faixa etária
dados['faixa_etaria'] = pd.cut(dados['idade'], bins=[30, 45, 60, 80],
                                labels=['30-45', '45-60', '60-80'])
print('Análise estratificada por faixa etária:')
for faixa in ['30-45', '45-60', '60-80']:
    sub = dados[dados['faixa_etaria'] == faixa]
    p_c = sub[sub['medicamento'] == 0]['complicacao'].mean()
    p_t = sub[sub['medicamento'] == 1]['complicacao'].mean()
    print(f'  {faixa}: controle={p_c:.3f}, tratamento={p_t:.3f}, diferença={p_t-p_c:.3f}')
print()
print('Dentro de cada faixa, o medicamento REDUZ o risco!')

## 💻 Exemplos Práticos

In [ ]:
# Exemplo: delineamento de um experimento agrícola (blocos randomizados)
import matplotlib.pyplot as plt

np.random.seed(42)

# Experimento: testar 3 fertilizantes em 4 blocos de solo (diferentes tipos)
blocos = ['Solo Arenoso', 'Solo Argiloso', 'Solo Orgânico', 'Solo Misto']
fertilizantes = ['A', 'B', 'C']

# Em cada bloco, sorteamos a ordem dos fertilizantes
resultados = {}
for bloco in blocos:
    # Efeito base do bloco
    efeito_base = np.random.uniform(20, 40)
    # Alocação aleatória dos fertilizantes dentro do bloco
    ordem = np.random.permutation(fertilizantes)
    resultados[bloco] = {}
    for i, fert in enumerate(ordem):
        efeito_fert = {'A': 5, 'B': 8, 'C': 3}[fert]
        produtividade = efeito_base + efeito_fert + np.random.normal(0, 2)
        resultados[bloco][fert] = round(produtividade, 1)

# Tabela de resultados
df = pd.DataFrame(resultados).T
print('Experimento em Blocos Randomizados — Produtividade (kg/parcela):')
print(df)
print()

# Média por fertilizante
print('Média por fertilizante:')
for fert in fertilizantes:
    medias = [resultados[b][fert] for b in blocos]
    print(f'  Fertilizante {fert}: {np.mean(medias):.1f} kg/parcela')

## ✅ Exercícios Resolvidos

**Exercício 1:** Um estudo observa que pessoas que bebem café têm menor incidência de diabetes tipo 2. Podemos concluir que café previne diabetes? Que outros fatores podem explicar essa associação?

In [ ]:
# Solução:
# Não podemos concluir causalidade apenas com estudo observacional.
# Possíveis fatores de confusão:
# 1. Pessoas que bebem café podem ter maior renda → acesso a melhor alimentação
# 2. Podem ser mais conscientes da saúde → praticam mais exercícios
# 3. Podem ter menor consumo de refrigerantes (substituição)
#
# Para estabelecer causalidade, precisaríamos de um ensaio clínico randomizado
# ou aplicar métodos como variáveis instrumentais ou propensity score matching.

print('Estudos observacionais geram hipóteses.')
print('Experimentos randomizados testam causalidade.')

## 📋 Exercícios Propostos

| Nível | Exercício |
|-------|-----------|
| 🟢 Fácil | 1. Dê um exemplo de pergunta que exige um estudo experimental e outra que pode ser respondida com estudo observacional. |
| 🟡 Médio | 2. Identifique possíveis fatores de confusão em um estudo que mostra que usuários de academias têm menor risco cardiovascular. |
| 🔴 Difícil | 3. Projete um experimento para testar se um novo método de ensino melhora notas em matemática, controlando fatores como nível socioeconômico e professor. |

In [ ]:
# 🟢 Exercício 1
# Experimental: 'Um novo medicamento reduz a pressão arterial?'
# → Ensaio clínico randomizado com grupo controle recebendo placebo
#
# Observacional: 'A poluição do ar está associada a doenças respiratórias?'
# → Estudo de coorte: acompanhar moradores de áreas com diferentes níveis de poluição

## 🏆 Desafios

1. Simule um estudo observacional com viés de confusão e mostre como a regressão múltipla pode controlá-lo
2. Implemente uma função que calcule o tamanho amostral necessário para um experimento com poder estatístico de 80%
3. Pesquise sobre o experimento de Milgram ou Stanford Prison Experiment e analise as questões éticas envolvidas

## 📌 Resumo
- **Estudos observacionais**: apenas observam — permitem associação, mas não causalidade
- **Estudos experimentais**: manipulam variáveis — permitem inferência causal
- **Randomização**: essencial para eliminar vieses de seleção
- **Grupo controle**: baseline para comparar o efeito do tratamento
- **Fatores de confusão**: variáveis que distorcem a verdadeira relação entre exposição e desfecho
- **Delineamento**: o plano do experimento deve ser definido antes da coleta de dados

## 💡 Curiosidades
- O primeiro ensaio clínico randomizado da história foi conduzido em 1946 pela British Medical Research Council para testar a estreptomicina contra tuberculose
- O estudo de Framingham (1948-presente) é um dos estudos de coorte mais famosos do mundo, responsável por estabelecer os fatores de risco cardiovasculares
- Em 1954, o experimento da vacina Salk contra poliomielite envolveu 1,8 milhão de crianças — um marco na história dos ensaios clínicos

## ✅ Boas Práticas
- Defina a pergunta de pesquisa antes de escolher o tipo de estudo
- Registre o protocolo do experimento antes de começar (pré-registro)
- Use randomização sempre que possível
- Inclua grupo controle adequado (não apenas comparar com 'antes')
- Documente todos os fatores de confusão potenciais
- Relate tanto resultados significativos quanto não-significativos

## ⚠️ Erros Comuns

| Erro | Como Evitar |
|------|-------------|
| Afirmar causalidade com base em estudo observacional | Use linguagem de associação: 'está associado a' em vez de 'causa' |
| Esquecer o grupo controle | Sempre inclua um grupo de comparação |
| Randomização inadequada | Use métodos robustos (não alternar manualmente) |
| Ignorar fatores de confusão | Liste e meça potenciais confundidores antes do estudo |

## 📖 Referências

- [W3Schools Statistics — Experiments](https://www.w3schools.com/statistics/statistics_experiments.php)
- [Khan Academy — Observational Studies](https://www.khanacademy.org/math/statistics-probability/designing-studies)
- [Wikipedia — Confounding](https://en.wikipedia.org/wiki/Confounding)

[◀ Anterior](S07_Parametros_Estatisticas.ipynb) | [📖 Índice](S00_Index_Estatistica.ipynb) | [Próximo ▶](S09_Tipos_de_Amostras.ipynb)

---
🧠 **Quer uma abordagem mais intuitiva?**
Visite o [companheiro de analogia: Tipos de Estudo — Vigilancia vs Experimento](S52_Tipos_de_Estudo_Analogias.ipynb)
para aprender este conteudo com uma historia do dia a dia.
